<a href="https://colab.research.google.com/github/ANEESAAYAZ/internship_Logistic_Regression/blob/main/Internship_LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

In [ ]:
df = pd.read_csv("properties_features.csv", parse_dates=["listed_date"])

In [ ]:
print(f"loaded {df.shape[0]} rows")

loaded 3000 rows


In [ ]:
settled = df[df["sold_fast"].notna()].copy()
settled["sold_fast"] = settled["sold_fast"].astype(int)

print(f"\nall listings        : {len(df)}")
print(f"still Active (dropped): {len(df) - len(settled)}")
print(f"modelling population  : {len(settled)}")
print(f"\nclass balance:")
print(settled["sold_fast"].value_counts().rename({0: "slow (0)", 1:"fast (1)"}))
print(f"positive rate: {settled['sold_fast'].mean():.1%}")


all listings        : 3000
still Active (dropped): 437
modelling population  : 2563

class balance:
sold_fast
slow (0)    1572
fast (1)     991
Name: count, dtype: int64
positive rate: 38.7%


In [ ]:
from sklearn.model_selection import train_test_split

FORBIDDEN = ["days_on_market", "sold_fast", "listing_status","price_reduced_count"]
DROP = FORBIDDEN + ["listed_date", "description", "latitude","longitude"]

X = settled.drop(columns=[c for c in DROP if c in settled.columns])
y = settled["sold_fast"]

print(f"\nfeatures available: {X.shape[1]}")
print("\nNOW LEGAL, because the price is already set:")
for c in ["price_aed", "log_price_aed", "price_per_sqft","rental_yield_pct", "area_price_encoded"]:
  print(f"  {c}")

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2, random_state=42, stratify=y)

print(f"\ntrain {len(X_train)} ({y_train.mean():.1%} positive)")
print(f"test  {len(X_test)} ({y_test.mean():.1%} positive)")


features available: 64

NOW LEGAL, because the price is already set:
  price_aed
  log_price_aed
  price_per_sqft
  rental_yield_pct
  area_price_encoded

train 2050 (38.7% positive)
test  513 (38.6% positive)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

def make_preprocessor():
  return ColumnTransformer([
      ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                        ("s", StandardScaler())]), numeric_cols),
      ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                        ("e", OneHotEncoder(handle_unknown="ignore",
                                            sparse_output=False,
                                            min_frequency=10))]), categorical_cols),
    ])
plain = Pipeline([("p", make_preprocessor()),
                  ("m", LogisticRegression(max_iter=3000))]).fit(X_train, y_train)

balanced = Pipeline([ ("p", make_preprocessor()),
                      ("m", LogisticRegression(max_iter=3000,class_weight="balanced"))]).fit(X_train, y_train)

print(f"\nfitted on {len(X_train)} rows x "
      f"{plain.named_steps['p'].transform(X_train).shape[1]} features")


fitted on 2050 rows x 177 features


In [ ]:
proba = balanced.predict_proba(X_test)[:, 1]
labels = balanced.predict(X_test)

print("\n--- what the model actually outputs ---")
print(f"  predict_proba shape {balanced.predict_proba(X_test).shape}  "
      f"(column 0 = P(slow), column 1 = P(fast))")
print(f" first 5 P(fast): {np.round(proba[:5], 3)}")
print(f" first 5 labels : {labels[:5]}")
print(f"\n  .predict() is just .predict_proba()[:,1] >= 0.5")
print(f"  agreement check: {np.array_equal(labels, (proba >= 0.5).astype(int))}")


--- what the model actually outputs ---
  predict_proba shape (513, 2)  (column 0 = P(slow), column 1 = P(fast))
 first 5 P(fast): [0.731 0.964 0.041 0.722 0.824]
 first 5 labels : [1 1 0 1 1]

  .predict() is just .predict_proba()[:,1] >= 0.5
  agreement check: True


**Evaluation**

In [15]:
from sklearn.metrics import (accuracy_score, average_precision_score,
                            classification_report, confusion_matrix, f1_score,
                            precision_score, recall_score, roc_auc_score)

always_slow = np.zeros(len(y_test), dtype=int)

print("\n--- three models, one metric ---")
print("="*42)
print(f" predict 'slow' for everything : accuracy {accuracy_score(y_test, always_slow):.3f}")
print(f" LogisticRegression (plain) : accuracy "
      f"{accuracy_score(y_test, plain.predict(X_test)):.3f}")
print(f"LogisticRegression (balanced) : accuracy {accuracy_score(y_test, labels):.3f}")
print("\n--- but look at what each one actually does ---")
print("="*42)
for name, pred in [("always slow", always_slow),
                  ("plain", plain.predict(X_test)),
                   ("balanced", labels)]:
  print(f"{name:12s} recall {recall_score(y_test, pred, zero_division=0):.3f}"
        f"   precision {precision_score(y_test, pred, zero_division=0):.3f}"
        f"   F1 {f1_score(y_test, pred, zero_division=0):.3f}")



--- three models, one metric ---
 predict 'slow' for everything : accuracy 0.614
 LogisticRegression (plain) : accuracy 0.795
LogisticRegression (balanced) : accuracy 0.788

--- but look at what each one actually does ---
always slow  recall 0.000   precision 0.000   F1 0.000
plain        recall 0.727   precision 0.738   F1 0.733
balanced     recall 0.778   precision 0.703   F1 0.739


In [17]:
cm = confusion_matrix(y_test, labels)
tn, fp, fn, tp = cm.ravel()
print("\n--- confusion matrix (balanced model, threshold 0.5) ---")
print(pd.DataFrame(cm,index=["actually slow", "actually fast"],
                  columns=["predicted slow", "predicted fast"]))
print(f"""
TN {tn:4d}  correctly left alone
FP {fp:4d}  we promised fast, it sold slowly  -> disappointed seller
FN {fn:4d}  we said slow, it sold fast  -> pointless price-cut advice
TP {tp:4d}  correctly flagged as fast movers""")


--- confusion matrix (balanced model, threshold 0.5) ---
               predicted slow  predicted fast
actually slow             250              65
actually fast              44             154
  
TN  250  correctly left alone  
FP   65  we promised fast, it sold slowly  -> disappointed seller  
FN   44  we said slow, it sold fast  -> pointless price-cut advice  
TP  154  correctly flagged as fast movers


In [18]:
print(classification_report(y_test, labels,target_names=["slow", "fast"], digits=3))

              precision    recall  f1-score   support

        slow      0.850     0.794     0.821       315
        fast      0.703     0.778     0.739       198

    accuracy                          0.788       513
   macro avg      0.777     0.786     0.780       513
weighted avg      0.794     0.788     0.789       513



In [19]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

auc = roc_auc_score(y_test, proba)
ap = average_precision_score(y_test, proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0], name="balanced")
axes[0].plot([0, 1], [0, 1], "k--",linewidth=1, label="random guessing")
axes[0].set_title(f"ROC curve (AUC = {auc:.3f})")
axes[0].legend()
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1], name="balanced")
axes[1].axhline( y_test.mean(), color="k", linestyle="--",
                linewidth=1,label=f"always-fast baseline ({y_test.mean():.2f})")
axes[1].set_title(f"Precision-Recall curve (AP = {ap:.3f})")
axes[1].legend()

plt.tight_layout()
plt.savefig("roc_pr_curves.png", dpi=110,bbox_inches="tight")
plt.close()

print(f"\nROC-AUC {auc:.4f} (0.5 = random, 1.0 = perfect)")
print(f"Average precision {ap:.4f}  (baseline = {y_test.mean():.4f}, the positive rate)")
print("saved roc_pr_curves.png")


ROC-AUC 0.8886 (0.5 = random, 1.0 = perfect)
Average precision 0.8410  (baseline = 0.3860, the positive rate)
saved roc_pr_curves.png


In [20]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, proba)

print("\n--- what different thresholds buy you ---")
print(f"{'threshold':>10s} {'precision':>10s} {'recall':>8s} {'flagged':>8s} {'F1':>7s}")
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
  p = (proba >= t).astype(int)
  print(f" {t:10.2f} {precision_score(y_test, p,zero_division=0):10.3f} "
        f"{recall_score(y_test, p, zero_division=0):8.3f} {p.sum():8d} "
        f"{f1_score(y_test, p, zero_division=0):7.3f}")


--- what different thresholds buy you ---
 threshold  precision   recall  flagged      F1
       0.20      0.616    0.939      302   0.744
       0.30      0.658    0.894      269   0.758
       0.40      0.676    0.823      241   0.743
       0.50      0.703    0.778      219   0.739
       0.60      0.736    0.732      197   0.734
       0.70      0.793    0.657      164   0.718
       0.80      0.867    0.495      113   0.630
